In [ ]:
# import numpy as np
import torch
import torch.nn as nn

# import torch.nn.functional as F
import torch.optim as optim
import torch.optim.lr_scheduler as lrs

from utils.dataset import get_cifar10_dataloaders, create_calibration_dataloader
from utils.resnet import quant_resnet18, filter_params
from utils.train import train_for_epoch

DATASET_PATH = "./data"  # Change me
EXPORT_PATH = "./model_outputs"  # Change me

In [ ]:
device = (
    "xpu"
    if torch.xpu.is_available()  # For the only person who has an Intel GPU
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

In [ ]:
# Training parameters
learning_rate_init = 0.1
learning_rate_step_size = 30
learning_rate_gamma = 0.1
momentum = 0.5
weight_decay = 1e-5

# Setup model
model = quant_resnet18(weight_bit_width=4, act_bit_width=4, num_classes=10)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    filter_params(model.named_parameters(), weight_decay),
    lr=learning_rate_init,
    weight_decay=weight_decay,
)
scheduler = lrs.StepLR(
    optimizer, step_size=learning_rate_step_size, gamma=learning_rate_gamma
)

In [ ]:
train_loader, test_loader = get_cifar10_dataloaders(
    DATASET_PATH, pin_memory=True, pin_memory_device=device
)
calib_loader = create_calibration_dataloader(dataset=train_loader.dataset)

In [ ]:
for epoch in range(20):  # 90 default
    train_loss = train_for_epoch(model, device, train_loader, criterion, optimizer)